<a href="https://colab.research.google.com/github/ankorn/redred/blob/main/redred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# seq2seq

In [ ]:
!pip uninstall datasets torchao

Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.12/dist-packages/datasets-3.6.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/datasets/*
Proceed (Y/n)? y
  Successfully uninstalled datasets-3.6.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchao/*
Proceed (Y/n)? y
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip -q install datasets==3.6.0 torchao==0.16.0 evaluate rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
from datasets import load_dataset

dataset = load_dataset("webis/tldr-17", split="train", streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-6-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # for BART; adjust for other models
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 334,053,376 || trainable%: 0.3531


In [ ]:
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["content"],
        max_length=1024, # CHECK: BART's max position embedding is 1024
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        examples["summary"],
        max_length=128, # TODO: CHECK
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

train_dataset = tokenized_dataset.shuffle().take(100000)
eval_dataset = tokenized_dataset.shuffle().take(1000)

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)

    # BERTScore (optional, slower)
    # from bert_score import score
    # P, R, F1 = score(decoded_preds, decoded_labels, lang="en")
    # result["bertscore"] = F1.mean().item()

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,
    metric_for_best_model="eval_rouge1",
    load_best_model_at_end=True,
    max_steps=1000
)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
100,8.934473,1.643555,0.135700,0.025000,0.097600,0.099300
200,8.073896,1.597656,0.134200,0.024500,0.096000,0.097700
300,7.962656,1.578125,0.134700,0.024200,0.096700,0.098500
400,7.610508,1.568359,0.135200,0.024700,0.096500,0.098600


KeyboardInterrupt: 

In [ ]:
post = '''Why do all old people have short hair?
I’ve noticed something and I’m curious if there’s a reason behind it. Most older women I see (like 70+) tend to have short hair. I honestly can’t think of many I’ve met with long hair.
Is there a practical reason for this? Like easier maintenance, hair thinning, or just preference over time? Would love to hear from anyone who knows or has experience with this.'''

In [ ]:
model.eval()
inputs = tokenizer(post, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

 Most older women (like 70+) tend to have short hair . Is there a practical reason for this? Like easier maintenance, hair thinning, or just preference over time? Would love to hear from anyone who knows or has experience with this. Would like to see anyone


# chat llm

In [1]:
%pip uninstall -q -y torchao

%pip -q install -q trl evaluate datasets==3.6.0 trl==0.12.0 bitsandbytes>=0.46.1 rouge_score peft torchao==0.16.0

In [2]:
import argparse
import json
import os
import random
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer
import evaluate
from tqdm import tqdm

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # ~1.1 GB Q4_K_M
# MODEL_NAME = "google/gemma-3-1b-it"       # ~600 MB Q4_K_M

MAX_SEQ_LENGTH = 1500

LORA_R = 64
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
WARMUP_RATIO = 0.03

In [4]:
def clean_text(text: str) -> str:
    text = text.replace("&#x200B;", "").replace("&gt;", ">").replace("&lt;", "<")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [5]:
dataset = load_dataset("webis/tldr-17", split="train", trust_remote_code=True, streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

tldr-17.py: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

In [6]:
from pandas import DataFrame
from datasets import Dataset

dataset = Dataset.from_pandas(DataFrame(list(dataset.take(300_000))))

dataset = dataset.train_test_split(test_size=0.005)
train_ds = dataset["train"]
eval_ds = dataset["test"]

In [7]:
system_msg = (
    "You are a helpful assistant that writes concise TL;DR summaries "
    "of Reddit posts in one or two sentences."
)

def format_chat(example):
    content = clean_text(example["content"])
    summary = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Reddit post:\n{content}"},
        {"role": "assistant", "content": summary},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
train_ds_formatted = train_ds.map(format_chat, remove_columns=train_ds.column_names)
eval_ds_formatted  = eval_ds.map(format_chat, remove_columns=eval_ds.column_names)

Map:   0%|          | 0/298500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [10]:
from peft import LoraConfig, TaskType, get_peft_model

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660


In [21]:
print(train_ds_formatted['text'][0])

<|im_start|>system
You are a helpful assistant that writes concise TL;DR summaries of Reddit posts in one or two sentences.<|im_end|>
<|im_start|>user
Reddit post:
There are some noticeable differences, such as the cleaner textures, but in terms of additional content, it's not much. There's a vault that contains all of the game audio and some art concepts from all three games as well as some DMC4 concept art. Unfortunately, however, the game does not come with better customization options. Like in-game brightness adjustments and full button mapping. There is also the fact that in DMC1 the subtitles are half-assed and not always on-screen when it should be, but that's my personal grief. I bought it a while ago and would recommend getting it if you want to have all three games on a single disc that runs better and is more easily accessible.<|im_end|>
<|im_start|>assistant
A vault with some music and art, and achievements/trophies.<|im_end|>



In [11]:
from trl import DataCollatorForCompletionOnlyLM

response_template = "<|im_start|>assistant\n"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template, 
    tokenizer=tokenizer
)

In [12]:
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_formatted,
    eval_dataset=eval_ds_formatted,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False, # False is safer for variable-length summaries
    data_collator=collator,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/298500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss
100,2.725788,2.776308
200,2.698774,2.752269
300,2.746119,2.739004
400,2.667114,2.736645


/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `<|im_start|>assistant
` in the following instance: <|im_start|>system
You are a helpful assistant that writes concise TL;DR summaries of Reddit posts in one or two sentences.<|im_end|>
<|im_start|>user
Reddit post:
I've told this story a few times on Reddit... But every time I see posts about the "humane" society, I just get angry all over again... And I have to tell the story.. I'm copying and pasting from the last time I posted it.. The top part might not be relevant because i don't feel like editing. So here's the Anyone who says spending that kind of money on a dog is a waste of money can go fuck themselves.. I've been to so many funerals (way too many...) for friends and family members.. But I've never cried harder than I did when my dogs have died. That shit is just utterly devastating. I'm gonna keep this story as short as possible, but I feel it's relevant.. I told it on 

KeyboardInterrupt: 

In [ ]:
import huggingface_hub

huggingface_hub.login()

In [ ]:
model.push_to_hub("pameydorke/redred-qwen2.5-1.5-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

CommitInfo(commit_url='https://huggingface.co/pameydorke/redred-summarizer/commit/3e3a1d9f0882ab944d3a16ba28365306489c7af8', commit_message='Upload model', commit_description='', oid='3e3a1d9f0882ab944d3a16ba28365306489c7af8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pameydorke/redred-summarizer', endpoint='https://huggingface.co', repo_type='model', repo_id='pameydorke/redred-summarizer'), pr_revision=None, pr_num=None)

In [ ]:
model.eval()

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

rouge = evaluate.load("rouge")
predictions, references = [], []

eval_subset = eval_ds.shuffle(seed=42).select(range(min(15, len(eval_ds))))

for example in tqdm(eval_subset, desc="Generating summaries"):
    content = clean_text(example["content"])
    ref = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Reddit post:\n{content}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
            length_penalty=1,
            # no_repeat_ngram_size=3,
        )
        
    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    predictions.append(pred)
    references.append(ref)

results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

for k, v in results.items():
    print(f"  {k}: {v:.4f}")

Generating summaries: 100%|██████████| 15/15 [02:34<00:00, 10.29s/it]


  rouge1: 0.1383
  rouge2: 0.0287
  rougeL: 0.0927
  rougeLsum: 0.0932


In [33]:
# length_penalty=0.8; no_repeat_ngram_size=None

print(predictions[0], '\n', references[0], '\n')
# print(predictions[42], '\n', references[42], '\n')
print(predictions[13], '\n', references[13], '\n')
print(predictions[11], '\n', references[11], '\n')
# print(predictions[39], '\n', references[39], '\n')

K-mart sucks. Edit: I'm sorry, I forgot to mention that I was in the women's section. The men's section was just as bad. Edit 2: It's not just me. I've seen this happen at other K-marts too. EDIT 3: I don't know if 
 Shopping at K-mart is like going back in time to the USSR. 

Rockstar North will continue to use this technique because it works. Other studios will use it too. Rockstar North is not the only studio that uses this technique. GTA 4 was the first game to use it. GTA V will be the second. GTA VI will probably be the third. GTA VII might be the 
 Tech is already developed, things get cheaper (especially cameras) 

Lannisters are the only ones who can stop him, and they don't seem to want to. Edit: Also, I'm not sure if this is the right place to ask this question, but is there a reason why the Lannister family doesn't want to stop him? Is it because they're afraid 
 He's Captain of the Kingsguard, and a Lannister at that, he can basically do whatever he pleases. 



In [ ]:
# initial: rouge1: 0.1373
# after 500 steps: rouge1: 0.1062

In [20]:
print(predictions[0], '\n', references[0], '\n')
print(predictions[42], '\n', references[42], '\n')
print(predictions[13], '\n', references[13], '\n')
print(predictions[11], '\n', references[11], '\n')
print(predictions[39], '\n', references[39], '\n')


Zarek saved us from the brink on the last turn. We almost lost due to an absence of food, but Zarek's once-a-game saved us from the brink on the last turn. We almost lost due to an absence of food, but Zarek's once-a-game saved us from the brink on the last turn. We almost lost due to an absence of food, but Zarek's once-a-game saved us from the brink on the last turn. We almost lost due to an absence of food, but Zarek's once-a-game saved us from the brink on the last turn. We almost lost 
 Zarek's finest moment, and why you should give a second thought to protestors 

I own a home. It isn't particularly large. It's two stories, maybe 3 or 4 bedrooms. I live far enough from the city that my neighbors' homes are not right up against mine, but close enough that I can drive there without too much trouble. It's a smaller town, but has everything I need in it. It is on the west coast, and the beach is nearby. Perhaps a 20 minute drive. There is almost no grass on the front. Just a cobbesto

In [ ]:
rouge = evaluate.load("rouge")
predictions, references = [], []

for example in tqdm(eval_subset):
    content = example["content"]   # keep raw; clean_text was not shown, apply if safe
    ref = example["summary"]

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Reddit post:\n{content}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            num_beams=1,               # simpler, greedy
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    if pred.startswith("assistant\n"):
        print('>>>', 'assistant on pred')
        pred = pred[len("assistant\n"):]

    if "<|im_end|>" in pred:
        print('>>>', '<|im_end|> on pred')
        pred = pred.replace("<|im_end|>", "").strip()

    predictions.append(pred)
    references.append(ref)

100%|██████████| 100/100 [08:03<00:00,  4.84s/it]


In [1]:
predictions[42], references[42]

NameError: name 'predictions' is not defined

In [ ]:
# 1. Merge LoRA into base model (do this before ONNX export)
merged_model = model.merge_and_unload()

# 2. Save merged model temporarily
import tempfile, os
temp_dir = tempfile.mkdtemp()
merged_model.save_pretrained(temp_dir)
tokenizer.save_pretrained(temp_dir)

# 3. Export to ONNX using Optimum
!pip install -q optimum[onnxruntime]

from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

onnx_temp = tempfile.mkdtemp()

# Export with cache for efficient inference
ort_model = ORTModelForCausalLM.from_pretrained(
    temp_dir,
    export=True,
    provider="CUDAExecutionProvider",  # or "CPUExecutionProvider"
    use_cache=True,
)

ort_model.save_pretrained(onnx_temp)
tokenizer.save_pretrained(onnx_temp)

# 4. Push to Hugging Face Hub
from huggingface_hub import HfApi, create_repo

repo_id = "pameydorke/redred-qwen2.5-1.5-lora-onnx" 
create_repo(repo_id, exist_ok=True, repo_type="model")

api = HfApi()
api.upload_folder(
    folder_path=onnx_temp,
    repo_id=repo_id,
    repo_type="model",
)

print(f"Pushed to https://huggingface.co/{repo_id}")

In [24]:
model.eval()

post_text = '''Why does Black Pepper get the place of honor next to Salt?
Salt is a substance that is literally necessary to keep us alive. Because of that, it makes sense that it is one of the five tastes that our tounge is able to detect. it basically just makes meat taste better on like an evolutionary level.

And then the thing that is always said in the same breath is just a seemingly random spice made from spicy ground up seeds or something. I assume that not all cultures value black pepper the same as we do in the west. But since it doesnt have the organic significance that salt does, I assume it could be any spicy thing we pair with salt. Why is it specifically black pepper?
'''

messages = [
    {"role": "system", "content": "Write a concise TL;DR summary of the Reddit post."},
    {"role": "user", "content": f"Reddit post:\n{post_text}"},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

summary = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
).strip()

summary

"Black pepper is not necessarily more flavorful than other spices because its flavor profile isn't particularly unique. The flavor profile of black pepper would probably be similar to cinnamon and cloves if you were to try them out yourself. They are both sweet, warm, and slightly spicier. So maybe they're not so different after all"

In [ ]:
adapted_path = 'https://huggingface.co/pameydorke/redred-summarizer'

In [ ]:
# (?)

peft_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = model.merge_and_unload()

from transformers import AutoModelForCausalLM
merged = AutoModelForCausalLM.from_pretrained("./tldr-lora-final")
merged.save_pretrained("./tldr-merged")

# 3. Convert to GGUF (install llama.cpp, then)
python convert_hf_to_gguf.py ./tldr-merged --outfile tldr-qwen-1.5b-q4_k_m.gguf --outtype q4_k_m

In [ ]:
# Merge your LoRA adapter into the base Qwen2.5-1.5B-Instruct.
# Export the merged model to ONNX with optimum-cli :
!optimum-cli export onnx --model ./tldr-merged ./tldr-onnx/
# Quantize for the browser:
!optimum-cli onnxruntime quantize --onnx_model ./tldr-onnx/ -o ./tldr-onnx-q4/

# JavaScript
# import { pipeline } from "@huggingface/transformers";
# const generator = await pipeline(
#   "text-generation",
#   "your-username/tldr-qwen-onnx",
#   { dtype: "q4", device: "webgpu" }
# );